In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import (classification_report, roc_auc_score, mean_squared_error, r2_score)
from sklearn.inspection import permutation_importance
import warnings; warnings.simplefilter('ignore')
from sklearn.datasets import make_classification

np.random.seed(42)

In [3]:
def ensemble_variance(sigma_sq,rho,n_trees):
  return rho*sigma_sq+(1-rho)*sigma_sq/n_trees

In [4]:
sigma_sq = 1.0
print("=== Ensemble Variance Reduction ===")
print(f"{'n_trees':>8} {'ρ=0.0 (ideal)':>16} {'ρ=0.3 (RF)':>14} {'ρ=0.9 (correlated)':>20}")
for n in [1, 5, 10, 50, 100, 500]:
    print(f"  {n:>6}   "
          f"{ensemble_variance(sigma_sq, 0.0, n):>14.4f}   "
          f"{ensemble_variance(sigma_sq, 0.3, n):>12.4f}   "
          f"{ensemble_variance(sigma_sq, 0.9, n):>18.4f}")

=== Ensemble Variance Reduction ===
 n_trees    ρ=0.0 (ideal)     ρ=0.3 (RF)   ρ=0.9 (correlated)
       1           1.0000         1.0000               1.0000
       5           0.2000         0.4400               0.9200
      10           0.1000         0.3700               0.9100
      50           0.0200         0.3140               0.9020
     100           0.0100         0.3070               0.9010
     500           0.0020         0.3014               0.9002


In [8]:
X, y = make_classification(n_samples=5000, n_features=12, n_informative=7,n_redundant=3, class_sep=0.8, random_state=42)
feature_names = [f"f{i}" for i in range(12)]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

### Basic Random Forest

In [9]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    max_features="sqrt",
    min_samples_leaf=5,
    oob_score=True,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:,1]

print(f"Random Forest Results:")
print(f"  OOB Score:     {rf.oob_score_:.4f}  (free estimate, no CV needed)")
print(f"  Test Accuracy: {rf.score(X_test, y_test):.4f}")
print(f"  AUC-ROC:       {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred))


Random Forest Results:
  OOB Score:     0.9085  (free estimate, no CV needed)
  Test Accuracy: 0.9110
  AUC-ROC:       0.9746
              precision    recall  f1-score   support

           0       0.90      0.92      0.91       500
           1       0.92      0.90      0.91       500

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000



### Feature Importance: MDI vs Permutation

In [10]:
mdi_importance = pd.Series(rf.feature_importances_, index=feature_names)


perm_result = permutation_importance(rf, X_test, y_test,
                                      n_repeats=10, random_state=42, n_jobs=-1)
perm_importance = pd.Series(perm_result.importances_mean, index=feature_names)

print("Feature Importance Comparison:")
print(f"{'Feature':>8} {'MDI':>10} {'Permutation':>14}")
for feat in feature_names[:8]:
    print(f"  {feat:>6}   {mdi_importance[feat]:>8.4f}   {perm_importance[feat]:>12.4f}")


Feature Importance Comparison:
 Feature        MDI    Permutation
      f0     0.0658         0.0182
      f1     0.1155         0.0674
      f2     0.1501         0.0883
      f3     0.0675         0.0086
      f4     0.0103        -0.0005
      f5     0.0684         0.0161
      f6     0.1162         0.0491
      f7     0.1015         0.0388


### Hyperparameter tuning with RandomizedSearchCV

In [11]:
param_dist = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [5, 8, 10, 15, None],
    "max_features":     ["sqrt", "log2", 0.3, 0.5],
    "min_samples_leaf": [1, 3, 5, 10, 20],
    "min_samples_split":[2, 5, 10],
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=42, oob_score=True),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
)
random_search.fit(X_train, y_train)
print(f"Best params (RandomizedSearch): {random_search.best_params_}")
print(f"Best CV AUC: {random_search.best_score_:.4f}")

Best params (RandomizedSearch): {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 0.3, 'max_depth': 15}
Best CV AUC: 0.9715


### RF Regression — Apartment Price Predictor

In [14]:
n = 3000
X_reg = np.column_stack([
    np.random.randint(800,4000,n),
    np.random.randint(1,6,n),
    np.random.uniform(1,10,n),
    np.random.randint(0,30,n),
    np.random.randint(0,2,n),
])
y_reg = (0.04*X_reg[:,0] + 5*X_reg[:,1] + 4*X_reg[:,2]
         - 0.5*X_reg[:,3] + 12*X_reg[:,4]
         + np.random.normal(0,15,n))

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=200, max_depth=10,
                                oob_score=True, n_jobs=-1, random_state=42)
rf_reg.fit(X_tr, y_tr)

y_pred_reg = rf_reg.predict(X_te)
print(f"RF Regression on Apartment Prices:")
print(f"  RMSE:      ₹{np.sqrt(mean_squared_error(y_te, y_pred_reg)):.1f}L")
print(f"  R²:        {r2_score(y_te, y_pred_reg):.4f}")
print(f"  OOB R²:    {rf_reg.oob_score_:.4f}")


tree_preds = np.array([tree.predict(X_te[:5]) for tree in rf_reg.estimators_])
lower = np.percentile(tree_preds, 5, axis=0)
upper = np.percentile(tree_preds, 95, axis=0)
point = rf_reg.predict(X_te[:5])
print(f"Point predictions + 90% intervals (first 5 test samples):")
for i in range(5):
    print(f"  ₹{point[i]:.1f}L  [₹{lower[i]:.1f}L – ₹{upper[i]:.1f}L]")

RF Regression on Apartment Prices:
  RMSE:      ₹15.5L
  R²:        0.8761
  OOB R²:    0.8523
Point predictions + 90% intervals (first 5 test samples):
  ₹165.7L  [₹151.2L – ₹194.4L]
  ₹132.4L  [₹117.9L – ₹151.7L]
  ₹143.0L  [₹124.6L – ₹168.2L]
  ₹202.5L  [₹187.7L – ₹230.2L]
  ₹67.7L  [₹50.1L – ₹91.5L]
